# 🧠 Modelo 5: Transfer Learning Avanzado con ResNet50

En este notebook utilizamos la arquitectura **ResNet50** (una de las más potentes en visión por computador) preentrenada en ImageNet. Aplicamos **Learning Rate Scheduling** y evaluamos su desempeño con métricas especializadas (AUC).

---

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
from sklearn.metrics import classification_report, confusion_matrix

# Añadir el directorio raíz al path para importar módulos locales
sys.path.append('..')
import oct_dataloader as dataloaders
import modelos.modelo_resnet50 as resnet_model

print("✅ Librerías e importaciones listas")

In [ ]:
# Configurar GPUs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU(s) detectada(s): {len(gpus)}")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ No se detectó GPU. Se usará la CPU.")

## 1. Carga de Datos (RGB para ResNet50)

**IMPORTANTE**: ResNet50 está diseñado para imágenes en color (3 canales). Por ello, configuramos `color_mode='rgb'` en el dataloader.

In [ ]:
# Descargar/Localizar dataset
path = kagglehub.dataset_download("anirudhcv/labeled-optical-coherence-tomography-oct")
data_path = path
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'test' in dirs:
        data_path = root
        break

# Configuración del DataLoader
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SUBSET = 0.3 # Usamos el 30% del entrenamiento para que sea fluido

train_ds, val_ds, test_ds, class_names = dataloaders.create_oct_dataloaders(
    data_path=data_path,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb', # <--- OBLIGATORIO para ResNet50
    train_subset_fraction=SUBSET, 
    optimize=True
)

## 2. Creación y Compilación del Modelo

ResNet50 tiene millones de parámetros. Congelamos su base para realizar solo **Transfer Learning** sobre las capas superiores que hemos añadido.

In [ ]:
from tensorflow.keras.metrics import AUC

# Hiperparámetros configurables
LEARNING_RATE = 0.001
DROPOUT = 0.4

# 1. Crear el modelo con ResNet50
model = resnet_model.create_resnet_model(
    input_shape=(128, 128, 3), # (Ancho, Alto, Canales RGB)
    num_classes=4, 
    dropout_rate=DROPOUT
)

# 2. Compilar especificando métricas (incluyendo el AUC para mayor robustez)
metrics = [
    'accuracy', 
    AUC(name='auc', multi_label=True)
]

model = resnet_model.compile_resnet_model(
    model, 
    learning_rate=LEARNING_RATE, 
    metrics=metrics
)

model.summary()

## 3. Entrenamiento con Scheduler

Implementamos `ReduceLROnPlateau` para mejorar la estabilidad del entrenamiento.

In [ ]:
EPOCHS = 15

# Obtener callbacks avanzados
callbacks = resnet_model.get_resnet_callbacks(
    patience_stop=5, 
    patience_lr=3, 
    factor_lr=0.2
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 4. Curvas de Aprendizaje y Evaluación

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history['auc'], label='Train')
plt.plot(history.history['val_auc'], label='Val')
plt.title('AUC')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()

plt.show()

In [ ]:
# Resultados finales en el conjunto de Test
print("📊 Evaluación Final (Test Set):")
results = model.evaluate(test_ds)
for i, metric in enumerate(model.metrics_names):
    print(f"{metric.capitalize()}: {results[i]:.4f}")